<a href="https://colab.research.google.com/github/Pradyun-8/esp32-tinyml-hazard-detector/blob/main/tinyml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd, numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

# ---- Load data (columns: gas, distance in cm) ----
df = pd.read_csv("sensor_readings.csv")
df.columns = ["gas", "dist"]

# Bad ultrasonic readings (negative or beyond 200 cm) are treated as "nothing nearby"
df["dist"] = df["dist"].where((df["dist"] >= 0) & (df["dist"] <= 200), 200)

# ---- Labels ----
GAS_LIMIT = 400    # above this = gas hazard
DIST_LIMIT = 25    # below this (cm) = obstacle

CLASSES = ["CLEAR", "OBSTACLE", "GAS", "GAS+OBSTACLE"]
gas_high = (df["gas"] > GAS_LIMIT).astype(int)
near = (df["dist"] < DIST_LIMIT).astype(int)
df["label"] = gas_high * 2 + near   # 0 CLEAR, 1 OBSTACLE, 2 GAS, 3 GAS+OBSTACLE

# ---- Train ----
X, y = df[["gas", "dist"]], df["label"]
model = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X, y)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)
print("train acc:", round(model.score(X, y), 3),
      " cv acc:", round(cross_val_score(model, X, y, cv=cv).mean(), 3))
print("samples per class:\n", df["label"].map(lambda i: CLASSES[i]).value_counts())

# ---- Export the trained tree as C code for the ESP32 ----
t = model.tree_

def emit(node, indent):
    pad = "  " * indent
    if t.feature[node] == -2:   # leaf
        cls = int(model.classes_[np.argmax(t.value[node])])
        return f"{pad}return {cls};  // {CLASSES[cls]}\n"
    name = ["gas", "dist"][t.feature[node]]
    s = f"{pad}if ({name} <= {t.threshold[node]:.2f}f) {{\n" + emit(t.children_left[node], indent + 1)
    return s + f"{pad}}} else {{\n" + emit(t.children_right[node], indent + 1) + f"{pad}}}\n"

print("\n// ---- Paste this into your ESP32 sketch ----")
print("int classify(float gas, float dist) {\n  if (dist < 0 || dist > 200) dist = 200;\n" + emit(0, 1) + "}")

train acc: 1.0  cv acc: 1.0
samples per class:
 label
OBSTACLE        53
CLEAR           52
GAS+OBSTACLE    52
GAS              2
Name: count, dtype: int64

// ---- Paste this into your ESP32 sketch ----
int classify(float gas, float dist) {
  if (dist < 0 || dist > 200) dist = 200;
  if (dist <= 25.50f) {
    if (gas <= 370.00f) {
      return 1;  // OBSTACLE
    } else {
      return 3;  // GAS+OBSTACLE
    }
  } else {
    if (gas <= 553.50f) {
      return 0;  // CLEAR
    } else {
      return 2;  // GAS
    }
  }
}


/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(
